In [ ]:
print('正在连接 Nexent 服务器并初始化环境...')
import os, sys, json, time, re, sqlite3, io, base64
from pathlib import Path
from IPython.display import display, Markdown, HTML, clear_output
import ipywidgets as widgets
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path(os.getcwd())
if ROOT.name == 'demo': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from clients.nexent_client import NexentClient

NEXENT_HOST = os.environ.get('CCF_NEXENT_CONFIG_BASE', 'https://nexent-api.mashiro.xin')
NEXENT_RUNTIME = os.environ.get('CCF_NEXENT_RUNTIME_BASE', 'https://nexent-runtime.mashiro.xin')
NEXENT_EMAIL = os.environ.get('CCF_NEXENT_EMAIL', 'suadmin@nexent.com')
# 演示 Notebook 默认使用已部署服务的演示账号；环境变量可覆盖默认值。
NEXENT_PASSWORD = os.environ.get('CCF_NEXENT_PASSWORD') or os.environ.get('NEXENT_PASSWORD') or '241002814'
TASK3_DEMO_URL = os.environ.get('CCF_TASK3_DEMO_URL', 'https://demo.mashiro.xin/')
client = NexentClient(NEXENT_HOST, NEXENT_RUNTIME, NEXENT_EMAIL, NEXENT_PASSWORD)
client.login()
agents = client.list_agents()
AGENT_IDS = {}
for a in agents:
    n = a.get('name','')
    if n == 'medical_data_cleaner': AGENT_IDS[1] = a['agent_id']
    elif n == 'medical_kg_qa': AGENT_IDS[2] = a['agent_id']
    elif n == 'medical_nl2sql': AGENT_IDS[3] = a['agent_id']

KG_DB = str(ROOT / 'data' / 'task2_medical_kg.db')
ANALYTICS_DB = str(ROOT / 'data' / 'task3_analytics.db')
EXAMPLE_DIR = ROOT / 'data' / 'demo_medical_texts'
EXAMPLE_FILES = sorted([f.name for f in EXAMPLE_DIR.glob('*.txt')]) if EXAMPLE_DIR.exists() else []

print(f'初始化完成。已连接 Nexent，发现 {len(AGENT_IDS)} 个Agent')
display(HTML(f'<div style="color:#059669;font-size:13px">Nexent 服务器已连接 | Agent 已就绪: {len(AGENT_IDS)} 个 | KG: {AGENT_IDS[1]} {AGENT_IDS[2]} {AGENT_IDS[3]}</div>'))

In [ ]:
print('正在加载样式和预设数据...')
TASK_LABELS = {1: '任务一：数据清洗', 2: '任务二：知识抽取', 3: '任务三：数据分析'}
EXAMPLES = {
    1: ['患者T2DM病史8年...主诉胸闷3天。用药：二甲双胍 1g bid + 阿卡波糖 50mg tid。',
        '【HIS导出】张三，男，45岁。T2DM，BP 180/110。SCr 128 umol/L。填表人：李护士。'],
    2: ['患者男65岁，胸痛2小时入院。胸骨后压榨样疼痛，向左肩放射。高血压病史10年。诊断急性前壁心肌梗死。',
        '45岁女性，持续性头痛伴恶心一周。2型糖尿病和高血压病史。口服二甲双胍和赖诺普利。'],
    3: ['糖尿病有哪些症状', '统计症状出现频率最高的前5项', '高血压用什么药',
        '当前知识图谱接入了多少数据来源', '百日咳应该挂什么科'],
}
CSS = """<style>
.ccf-chat{max-width:960px;margin:0 auto;font-family:-apple-system,BlinkMacSystemFont,"Microsoft YaHei",sans-serif}
.ccf-user{text-align:right;margin:10px 0}
.ccf-user span{background:#1d4ed8;color:#fff;padding:8px 14px;border-radius:14px 14px 2px 14px;display:inline-block;max-width:78%;font-size:14px;line-height:1.6;word-break:break-word}
.ccf-agent{margin:10px 0}
.ccf-agent .body{background:#f1f5f9;padding:10px 15px;border-radius:2px 14px 14px 14px;display:inline-block;max-width:85%;font-size:14px;line-height:1.7}
.ccf-tool{display:inline-block;background:#dbeafe;color:#1e40af;padding:1px 7px;border-radius:6px;font-size:11px;margin:2px}
.ccf-detail{cursor:pointer;color:#64748b;font-size:12px;margin:4px 0}
.ccf-kpi{display:inline-block;background:#fff;border-radius:8px;padding:14px 20px;margin:8px;box-shadow:0 1px 2px rgba(0,0,0,.06);text-align:center;min-width:130px}
.ccf-kpi .val{font-size:26px;font-weight:700;color:#0f172a}
.ccf-kpi .lbl{font-size:11px;color:#94a3b8;margin-top:2px}
.ccf-h2{font-size:17px;font-weight:600;color:#0f172a;margin:20px 0 6px 0;padding-bottom:4px;border-bottom:2px solid #1d4ed8}
.ccf-hint{color:#64748b;font-size:12px;margin:2px 0}
.ccf-link{color:#1d4ed8;text-decoration:none;font-size:12px}
</style>"""
display(HTML(CSS))
print('样式加载完成')

In [ ]:
print('正在构建交互界面...')
chat_out = widgets.Output()
uploader = widgets.FileUpload(accept='.txt,.csv,.json', multiple=False, layout=widgets.Layout(width='100%'))
text_input = widgets.Textarea(
    placeholder='输入医疗文本或问题，点击发送开始对话...',
    layout=widgets.Layout(width='100%', height='80px')
)
send_btn = widgets.Button(description='发送', button_style='primary', layout=widgets.Layout(width='80px', height='36px'))
task_dd = widgets.Dropdown(
    options=[('任务一：数据清洗', 1), ('任务二：知识抽取', 2), ('任务三：数据分析', 3)],
    value=3, description='Agent：', layout=widgets.Layout(width='240px')
)
example_dd = widgets.Dropdown(options=[], description='示例问题：', layout=widgets.Layout(width='540px'))
upload_info = widgets.HTML('')
input_tabs = widgets.Tab()

def _refresh_examples(change=None):
    ex = EXAMPLES[task_dd.value]
    example_dd.options = [(e[:85] + ('...' if len(e)>85 else ''), e) for e in ex]
    if ex: example_dd.value = ex[0]
task_dd.observe(_refresh_examples, 'value')
_refresh_examples()

def _on_upload(change):
    if uploader.value:
        k = list(uploader.value.keys())[0]
        txt = list(uploader.value.values())[0]['content'].decode('utf-8', errors='replace')
        text_input.value = txt[:5000]
        upload_info.value = f'<span class="ccf-hint">已加载: {k} ({len(txt)} 字符)</span>'
uploader.observe(_on_upload, 'value')

def _send(_):
    q = text_input.value.strip()
    if not q: return
    task = task_dd.value
    with chat_out:
        display(HTML(f'<div class="ccf-user"><span>{q[:300]}</span></div>'))
        box = widgets.Output()
        display(box)
        chunks, tools = [], []
        update_count = 0
        try:
            for ev in client.run_agent_stream(AGENT_IDS[task], q):
                t = ev.get('type',''); c = str(ev.get('content',''))
                if t == 'parse': tools.append(c[:120])
                elif t in ('model_output_thinking','final_answer'):
                    chunks.append(c); update_count += 1
                    if update_count % 8 == 0 or t == 'final_answer':
                        with box: clear_output(wait=True); display(HTML(f'<div class="ccf-agent"><div class="body">{Markdown("".join(chunks[:3000])).data}</div></div>'))
        except Exception as e:
            with box: display(HTML(f'<div style="color:#dc2626">{e}</div>'))
        with box: clear_output(wait=True); display(HTML(f'<div class="ccf-agent"><div class="body">{Markdown("".join(chunks[:3000])).data}</div></div>'))
        if tools:
            chips = ''.join(f'<span class="ccf-tool">{i+1}. {t[:80]}</span>' for i, t in enumerate(tools))
            with box: display(HTML(f'<details class="ccf-detail"><summary>工具调用记录 ({len(tools)} 次)</summary>{chips}</details>'))
    text_input.value = ''

def _use_example(_): text_input.value = example_dd.value

send_btn.on_click(_send)
text_input.continuous_update = False
text_input.observe(lambda _: setattr(send_btn, 'disabled', not text_input.value.strip()), 'value')

example_btn = widgets.Button(description='使用此示例', button_style='info', layout=widgets.Layout(width='100px', height='36px'))
example_btn.on_click(_use_example)

demo_sel = widgets.Select(options=EXAMPLE_FILES, layout=widgets.Layout(width='100%', height='120px')) if EXAMPLE_FILES else widgets.HTML('<p class="ccf-hint">data/demo_medical_texts/ 下暂无示例文件</p>')

def _load_demo(_):
    if EXAMPLE_FILES:
        txt = (EXAMPLE_DIR / demo_sel.value).read_text(encoding='utf-8', errors='replace')[:5000]
        text_input.value = txt
        upload_info.value = f'<span class="ccf-hint">已加载: {demo_sel.value}</span>'
if EXAMPLE_FILES: demo_sel.observe(_load_demo, 'value')

input_tabs.children = [
    widgets.VBox([text_input, widgets.HBox([send_btn])]),
    widgets.VBox([widgets.HTML('<p class="ccf-hint">支持 .txt / .csv / .json 文件</p>'), uploader, upload_info]),
    widgets.VBox([widgets.HTML('<p class="ccf-hint">从项目自带的示例文件中选择，选中即加载</p>'), demo_sel, upload_info]),
]
input_tabs.set_title(0, '文字输入')
input_tabs.set_title(1, '上传文件')
input_tabs.set_title(2, '示例文件')

print('界面就绪，可以开始对话')
display(HTML('<div class="ccf-h2">医疗AI智能体交互</div>'))
display(widgets.HBox([task_dd, example_dd, example_btn]))
display(input_tabs)
display(chat_out)

In [ ]:
print('正在查询知识图谱和分析数据库...')
def _q(sql, db=ANALYTICS_DB):
    c = sqlite3.connect(db); df = pd.read_sql_query(sql, c); c.close(); return df

display(HTML('<div class="ccf-h2">知识图谱与数据分析看板</div>'))

ent = _q('SELECT count(*) FROM kg_entities', KG_DB).iloc[0,0]
tri = _q('SELECT count(*) FROM kg_triples', KG_DB).iloc[0,0]
dis = _q('SELECT count(*) FROM diseases').iloc[0,0]
sym = _q('SELECT count(*) FROM disease_symptoms').iloc[0,0]
display(HTML(f'<div class="ccf-kpi"><div class="val">{ent//1000}K</div><div class="lbl">实体</div></div><div class="ccf-kpi"><div class="val">{tri//1000}K</div><div class="lbl">三元组</div></div><div class="ccf-kpi"><div class="val">{dis//1000}K</div><div class="lbl">疾病</div></div><div class="ccf-kpi"><div class="val">{sym//1000}K</div><div class="lbl">症状记录</div></div>'))

print('正在渲染图表...')
display(HTML('<div style="font-weight:600;margin:16px 0 4px">症状关联疾病数 Top 10</div>'))
df = _q('SELECT symptom, count(*) cnt FROM disease_symptoms GROUP BY symptom ORDER BY cnt DESC LIMIT 10')
fig = px.bar(df, x='symptom', y='cnt', text_auto=True, height=350, color_discrete_sequence=['#1d4ed8'])
fig.update_layout(margin=dict(t=10,b=0,l=0,r=0), plot_bgcolor='#fff', paper_bgcolor='#fff')
fig.update_traces(textposition='outside')
fig.show()

display(HTML('<div style="font-weight:600;margin:16px 0 4px">疾病科室分布</div>'))
df2 = _q('SELECT department, count(*) cnt FROM disease_departments GROUP BY department ORDER BY cnt DESC LIMIT 8')
fig2 = px.pie(df2, names='department', values='cnt', height=350, color_discrete_sequence=px.colors.qualitative.Set2)
fig2.update_layout(margin=dict(t=10,b=0,l=0,r=0), paper_bgcolor='#fff')
fig2.show()

In [ ]:
display(HTML('<div class="ccf-h2">全链路演示：清洗 - 抽取 - 查询</div>'))
display(HTML('<p class="ccf-hint">依次调用三个Agent，展示从原始文本到知识图谱再到分析查询的完整流程。清洗数据取自上方输入框，如为空则使用默认示例。</p>'))

pipe_text = text_input.value.strip() or EXAMPLES[1][0]
pipe_extract = EXAMPLES[2][0]
pipe_query = EXAMPLES[3][0]

steps = [
    (1, '任务一：数据清洗', '请清洗以下医疗文本：' + pipe_text),
    (2, '任务二：知识抽取', '请抽取实体、关系和三元组：' + pipe_extract),
    (3, '任务三：数据分析', pipe_query),
]

for tid, lbl, q in steps:
    display(HTML(f'<div style="font-weight:600;margin:8px 0">{lbl}</div>'))
    out = widgets.Output()
    display(out)
    chunks, tc = [], 0
    try:
        for ev in client.run_agent_stream(AGENT_IDS[tid], q):
            t = ev.get('type',''); c = str(ev.get('content',''))
            if t == 'parse': tc += 1
            elif t in ('model_output_thinking','final_answer'): chunks.append(c)
        result = ''.join(chunks)[:1500]
        with out:
            display(HTML(f'<div style="background:#f8fafc;padding:10px;border-radius:6px;font-size:13px;max-height:220px;overflow-y:auto">{Markdown(result).data}</div>'))
            display(HTML(f'<span style="color:#059669;font-size:11px">完成 | {tc} 次工具调用 | <a class="ccf-link" href="{TASK3_DEMO_URL.rstrip('/')}" target="_blank">可视化入口</a></span>'))
    except Exception as e:
        with out: display(HTML(f'<span style="color:#dc2626">{e}</span>'))
    display(HTML('<hr style="border-color:#e2e8f0;margin:4px 0">'))